# QianfanOCR - 版面分析（Layout Analysis）

识别文档图片中的所有版面元素，输出每个元素的边界框（bbox）和类别（category）。坐标归一化到 `[0, 1000)` 范围内。

支持的类别共 **25** 种：

| 类别 | 说明 | 类别 | 说明 |
|------|------|------|------|
| `abstract` | 摘要 | `header_image` | 页眉图片 |
| `algorithm` | 算法块 | `image` | 图片 |
| `aside_text` | 旁注文字 | `inline_formula` | 行内公式 |
| `chart` | 图表 | `number` | 页码 |
| `content` | 目录 | `paragraph_title` | 段落标题 |
| `display_formula` | 行间公式 | `reference` | 参考文献标题 |
| `doc_title` | 文档标题 | `reference_content` | 参考文献内容 |
| `figure_title` | 图片标题 | `seal` | 印章 |
| `footer` | 页脚 | `table` | 表格 |
| `footer_image` | 页脚图片 | `text` | 文本 |
| `footnote` | 脚注 | `vertical_text` | 竖排文字 |
| `formula_number` | 公式编号 | `vision_footnote` | 图注 |
| `header` | 页眉 | | |

## 1. 环境准备

In [ ]:
import re
import json
import base64
import requests
from PIL import Image, ImageDraw, ImageFont

## 2. 配置参数

请将 `API_URL` 和 `API_KEY` 替换为实际的服务地址和密钥。

In [ ]:
API_URL = "http://your-api-endpoint/v1/chat/completions"
API_KEY = "your-api-key"

# 模型超参
MODEL_NAME = "ocrfp8"
MAX_TOKENS = 4096
SKIP_SPECIAL_TOKENS = False

## 3. 读取图片

In [ ]:
image_path = "../images/paper.png"

with open(image_path, "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode("utf-8")

print(f"图片已加载: {image_path}")

## 4. 调用版面分析 API

In [ ]:
LAYOUT_PROMPT = """Extract all layout elements from this image.
Output in JSON format. Each element must include:
1.  `bbox`: `[x1, y1, x2, y2]`
2.  `category`: One of ['abstract', 'algorithm', 'aside_text', 'chart', 'content', 'display_formula', 'inline_formula', 'doc_title', 'figure_title', 'footer', 'footer_image', 'footnote', 'formula_number', 'header', 'header_image', 'image', 'number', 'paragraph_title', 'reference', 'reference_content', 'seal', 'table', 'text', 'vertical_text', 'vision_footnote']

**Important**: Do not include any extracted text in the output."""

payload = {
    "model": MODEL_NAME,
    "max_tokens": MAX_TOKENS,
    "mm_processor_kwargs": {
        "skip_special_tokens": SKIP_SPECIAL_TOKENS,
    },
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_base64}"
                    }
                },
                {
                    "type": "text",
                    "text": LAYOUT_PROMPT
                }
            ]
        }
    ]
}

response = requests.post(
    API_URL,
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}"
    },
    json=payload
)

model_output = response.json()["choices"][0]["message"]["content"]
print(model_output)

## 5. 解析模型输出

> **坐标说明**：`<COORD_N>` 中的 `N` 即为坐标值，归一化到 `[0, 1000)` 范围。例如 `<COORD_159>` 表示归一化坐标 `159`。还原到像素坐标的公式为：
>
> $$x_{\text{pixel}} = \frac{\text{COORD}}{1000} \times W, \quad y_{\text{pixel}} = \frac{\text{COORD}}{1000} \times H$$
>
> 其中 $W$、$H$ 分别为原图的宽度和高度（像素）。

In [ ]:
def parse_layout_output(raw_output: str) -> list[dict]:
    """
    解析模型输出的版面分析结果。

    支持两种坐标格式：
      - Token 格式：<COORD_159> → 159
      - 纯数字格式：159
    """
    # 将 <COORD_N> 替换为数字 N
    cleaned = re.sub(r"<COORD_(\d+)>", r"\1", raw_output)
    elements = json.loads(cleaned)

    parsed = []
    for elem in elements:
        bbox_raw = elem["bbox"]
        # bbox 可能是字符串 "[x1, y1, x2, y2]" 或已解析的列表
        if isinstance(bbox_raw, str):
            bbox = [int(x) for x in re.findall(r"\d+", bbox_raw)]
        else:
            bbox = bbox_raw
        parsed.append({
            "bbox": [int(v) for v in bbox],
            "category": elem["category"],
        })
    return parsed


elements = parse_layout_output(model_output)
print(f"检测到 {len(elements)} 个版面元素:")
for elem in elements:
    print(f"  [{elem['category']:>20s}]  bbox={elem['bbox']}")

## 6. 可视化版面分析结果

将版面分析结果绘制到原图上，每种颜色对应一个版面类别。

In [ ]:
CATEGORY_COLORS = {
    "abstract":          "#FF6B6B",
    "algorithm":         "#4ECDC4",
    "aside_text":        "#45B7D1",
    "chart":             "#96CEB4",
    "content":           "#FFEAA7",
    "display_formula":   "#DDA0DD",
    "doc_title":         "#FF4757",
    "inline_formula":    "#B33771",
    "figure_title":      "#2ED573",
    "footer":            "#A4B0BE",
    "footer_image":      "#747D8C",
    "footnote":          "#FECA57",
    "formula_number":    "#FF9FF3",
    "header":            "#54A0FF",
    "header_image":      "#5F27CD",
    "image":             "#FF6348",
    "number":            "#1DD1A1",
    "paragraph_title":   "#F368E0",
    "reference":         "#EE5A24",
    "reference_content": "#FFC312",
    "seal":              "#C44569",
    "table":             "#3DC1D3",
    "text":              "#E77F67",
    "vertical_text":     "#786FA6",
    "vision_footnote":   "#F8A5C2",
}


def visualize_layout(
    image_path: str,
    layout_elements: list[dict],
    output_path: str | None = None,
    line_width: int = 3,
    font_size: int = 16,
    fill_alpha: int = 40,
) -> Image.Image:
    """将版面分析结果可视化到原图上。"""
    img = Image.open(image_path).convert("RGBA")
    W, H = img.size

    overlay = Image.new("RGBA", img.size, (255, 255, 255, 0))
    draw_overlay = ImageDraw.Draw(overlay)
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", font_size)
    except (IOError, OSError):
        font = ImageFont.load_default()

    for elem in layout_elements:
        x1_norm, y1_norm, x2_norm, y2_norm = elem["bbox"]
        category = elem["category"]

        x1 = int(x1_norm / 1000 * W)
        y1 = int(y1_norm / 1000 * H)
        x2 = int(x2_norm / 1000 * W)
        y2 = int(y2_norm / 1000 * H)

        hex_color = CATEGORY_COLORS.get(category, "#CCCCCC")
        r, g, b = int(hex_color[1:3], 16), int(hex_color[3:5], 16), int(hex_color[5:7], 16)

        draw_overlay.rectangle([x1, y1, x2, y2], fill=(r, g, b, fill_alpha))
        draw.rectangle([x1, y1, x2, y2], outline=(r, g, b, 255), width=line_width)

        label = category
        text_bbox = font.getbbox(label)
        text_w = text_bbox[2] - text_bbox[0]
        text_h = text_bbox[3] - text_bbox[1]
        label_x = x1
        label_y = max(y1 - text_h - 4, 0)
        draw.rectangle(
            [label_x, label_y, label_x + text_w + 6, label_y + text_h + 4],
            fill=(r, g, b, 255),
        )
        draw.text((label_x + 3, label_y + 1), label, fill=(255, 255, 255, 255), font=font)

    img = Image.alpha_composite(img, overlay).convert("RGB")

    if output_path:
        img.save(output_path)
        print(f"可视化结果已保存至: {output_path}")

    return img

In [ ]:
result_img = visualize_layout(
    image_path=image_path,
    layout_elements=elements,
    output_path="page_visualized.png",
)
result_img